## Stage 1: Online Softmax

In [10]:
import torch

In [11]:
torch.manual_seed(0)

In [12]:
N = 10 # number of keys (length of one score row)
d = 4 # size of each value vector

In [13]:
scores = torch.randn(N, dtype=torch.float64)  # one query's scores against all N keys
values = torch.randn(N, d, dtype=torch.float64)  # one value vector per key

In [14]:
# Reference: ordinary softmax over the whole row at once
probs = torch.softmax(scores, dim=0) # (N,)
out_ref = probs @ values  # (d,)

In [15]:
probs.shape, out_ref.shape

(torch.Size([10]), torch.Size([4]))

In [16]:
block = 3

m = torch.tensor(float("-inf"), dtype=torch.float64) # running max of scores seen so far
l = torch.tensor(0.0, dtype=torch.float64) # running sum of exp(score - m) -> SOFTMAX DENOMINATOR
acc = torch.zeros(d, dtype=torch.float64) # running sum of exp(score - m) * value -> SOFTMAX NUMERATOR

In [17]:
"""
Everything in l and acc so far was computed as exp(score - m_old). 
If this block has a larger max, those old terms are relative to the wrong max. 
Since exp(s - m_new) = exp(s - m_old) · exp(m_old - m_new), multiplying the old totals by alpha puts them on the new max.

If the max didn't change, alpha = exp(0) = 1, and nothing is rescaled.
On the first block, alpha = exp(-inf) = 0, which multiplies the old totals by zero. They're already zero, so this is harmless.
"""

"\nEverything in l and acc so far was computed as exp(score - m_old). \nIf this block has a larger max, those old terms are relative to the wrong max. \nSince exp(s - m_new) = exp(s - m_old) · exp(m_old - m_new), multiplying the old totals by alpha puts them on the new max.\n\nIf the max didn't change, alpha = exp(0) = 1, and nothing is rescaled.\nOn the first block, alpha = exp(-inf) = 0, which multiplies the old totals by zero. They're already zero, so this is harmless.\n"

In [18]:
for start in range(0, N, block):
    end = min(start + block, N)
    s_blk = scores[start:end] # (block,)
    v_blk = values[start:end] # (block, d)

    m_new = torch.maximum(m, s_blk.max())
    alpha = torch.exp(m - m_new) # rescales old totals from old max to new max 
    p = torch.exp(s_blk - m_new) # this blocks unnormalized weights, (block, )

    l = alpha * l + p.sum() # update running sum of exp(score - m)
    acc = alpha * acc + p @ v_blk # (d,)
    m = m_new

out = acc / l # normalize by the running sum of exp(score - m)
torch.testing.assert_close(out, out_ref)
print("online softmax matches:", out)

online softmax matches: tensor([ 0.3549,  0.0543, -0.1072,  0.2934], dtype=torch.float64)


## Stage 2: All queries at once, looping over K/V blocks

In [19]:
import math
import torch

torch.manual_seed(0)

N = 10  # sequence length
d = 4  # head dimension 

Q = torch.randn(N, d, dtype=torch.float64)
K = torch.randn(N, d, dtype=torch.float64)
V = torch.randn(N, d, dtype=torch.float64)

# Reference: standard attention, builds the full (N, N) matrix
scores = Q @ K.T / math.sqrt(d) # (N, N)
probs = torch.softmax(scores, dim=-1) # (N, N)
out_ref = probs @ V # (N, d)

In [20]:
block_k = 3

m = torch.full((N,), float('-inf'), dtype=torch.float64) # (N,) running max for each query row
l = torch.zeros(N, dtype=torch.float64) # (N,) running sum for each query row
acc = torch.zeros(N, d, dtype=torch.float64) # (N, d) running weighted sum of values

In [21]:
for start in range(0, N, block_k):
    end = min(start + block_k, N)
    k_blk = K[start:end] # (block_k, d) current block of keys
    v_blk = V[start:end] # (block_k, d) current block of values

    s_blk = Q @ k_blk.T / math.sqrt(d) # (N, block_k) current block of scores
    m_new = torch.maximum(m, s_blk.amax(dim=-1)) # (N,) updated running max for each query row
    alpha = torch.exp(m - m_new) # (N,) scaling factor for the running sum
    p = torch.exp(s_blk - m_new[:, None]) # (N, block_k) normalized probabilities for the current block

    l = alpha * l + p.sum(dim=-1) # (N,)
    acc = alpha[:, None] * acc + p @ v_blk # (N, d)
    m = m_new # update running max for the next block

out = acc / l[:, None] # (N, d)
torch.testing.assert_close(out, out_ref)
print("stage 2 matches")

stage 2 matches


## Stage 3: Split the queries into blocks

In [22]:
block_q = 4
block_k = 3

O = torch.zeros(N, d, dtype=torch.float64) # final output, filled one Q block at a time

In [23]:
for q_start in range(0, N, block_q):
    q_end = min(q_start + block_q, N)
    Q_blk = Q[q_start:q_end] # (block_q, d) current block of queries
    bq = q_end - q_start # (block_q,) size of the current query block

    m = torch.full((bq,), float("-inf"), dtype=torch.float64) # (bq,)
    l = torch.zeros(bq, dtype=torch.float64) # (bq,)
    acc = torch.zeros(bq, d, dtype=torch.float64) # (bq, d)

    for k_start in range(0, N, block_k):
        k_end = min(k_start + block_k, N)
        K_blk = K[k_start:k_end]                   # (bk, d)
        V_blk = V[k_start:k_end]                   # (bk, d)

        s_blk = Q_blk @ K_blk.T / math.sqrt(d)     # (bq, bk)  <- one tile of the score matrix

        m_new = torch.maximum(m, s_blk.amax(dim=-1))   # (bq,)
        alpha = torch.exp(m - m_new)                   # (bq,)
        p = torch.exp(s_blk - m_new[:, None])          # (bq, bk)

        l = alpha * l + p.sum(dim=-1)                  # (bq,)
        acc = alpha[:, None] * acc + p @ V_blk         # (bq, d)
        m = m_new
    
    O[q_start:q_end] = acc / l[:, None] # (bq, d)

torch.testing.assert_close(O, out_ref)
print("stage 3 matches")


stage 3 matches
